# Data Analysis with Real Polygraphs Data

## Introduction

Welcome to the final session of the Human Network Sciences Workshop! In this session, we'll be diving into data analysis using real data from the Polygraphs framework. We'll build on the concepts you've learned in the previous sessions and apply them to analyze network structures, dynamics, and statistical properties.

Our goals for this session are:
- Learn how to load and prepare Polygraphs simulation data for analysis.
- Explore basic network properties and visualize key metrics.
- Perform statistical analyses to compare different network configurations.
- Understand how to interpret the results.

Our session is divided into three main parts:
1. Data Indexing and Processing
2. Exploratory Data Analysis
3. Statistical Analysis


In [1]:
import os
import json
import networkx as nx
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import h5py
import scipy.stats as stats
from tqdm import tqdm
import scipy.stats as stats
from scipy.stats import chi2_contingency
from scipy.stats import f_oneway
import warnings
from concurrent.futures import ProcessPoolExecutor
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
import itertools

# More readable way to view large dataframes in ipython
from IPython.display import display



In [ ]:
# Installing Polygraphs
try:
    import polygraphs
    print("Polygraphs is already installed.")
except ImportError:
    print("Polygraphs is not installed. Installing now...")
    !pip install torch==2.4.1 torchdata
    !pip install  dgl -f https://data.dgl.ai/wheels/torch-2.4/repo.html
    !pip install polygraphs
    !python -m dgl.backend.set_default_backend . pytorch

import torch
import dgl
torch.cuda.is_available()

# Importing Polygraphs modules
import polygraphs
from polygraphs.analysis import Processor
from polygraphs.ops.common import BalaGoyalOp
from polygraphs.ops.complex import UnreliableNetworkBasicGullibleUniformOp
print("Polygraphs modules imported successfully.")


# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

Polygraphs is not installed. Installing now...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.1/797.1 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.2/176.2 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.1/99.1 kB 8.1 MB/s eta 0:00:

In [ ]:
try:
    import igraph as ig
except ImportError:
    !pip install python-igraph
    import igraph as ig


## Part 1: Data Indexing and Processing

In this section, we'll load our simulation data and create an indexer to help us organize and analyze the results efficiently.


### Loading Simulation Data

First, let's use the Polygraphs Processor to extract our simulation data.

This might take a couple of minutes, as the data files are large.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


ValueError: mount failed

Open Google Drive in your browser (navigate to **drive.google.com**).

Click on "**Shared with** me" in the left-hand menu. This is where you can see all files and folders that have been shared with you.

Locate the shared folder (e.g., "**Structure Meets Strategy Data**").

Right-click on the folder (or, on a Mac, Control-click), go to "Organize", and then select "**Add shortcut to Drive**" from the dropdown menu.

In the pop-up window, navigate to "**My Drive**" (or the specific location you want to place the shortcut).

Click the "**Add Shortcut**" button to confirm.


In [ ]:
data_path = r"/content/drive/My Drive/HNS_Polygraphs_Data"

# Check if the path exists
if os.path.exists(data_path):
    print(f"Folder found: {data_path}")
    # List contents of the folder
    print("Contents of the folder:")
    print(os.listdir(data_path))
else:
    print("Data folder not found, please double-check the path.")


This function takes a folder path (as input) and looks through that folder and its subfolders to find specific files like a configuration.json file and a data.csv file.

It reads these files, combines the data, and returns a dataframe that has all the useful information.



In [ ]:
def manual_processing(path):
    path = Path(path)
    folders = [path, *[x for x in path.rglob("*/")]]
    result_df = pd.DataFrame(columns=["bin_file_path", "hd5_file_path", "config_json_path"])

    for folder in folders:
        config_path = folder / 'configuration.json'
        data_path = folder / 'data.csv'

        if config_path.exists() and data_path.exists():
            try:
                with open(config_path, 'r') as f:
                    config = json.load(f)

                df = pd.read_csv(data_path)

                df['network_kind'] = config.get('network', {}).get('kind', 'unknown')
                df['config_json_path'] = str(config_path)

                bin_file_name = config.get('graph', {}).get('binary_path')
                hd5_file_name = config.get('graph', {}).get('hd5_path')

                if not bin_file_name:
                    bin_files = [f for f in folder.glob('*.bin')]
                    bin_file_name = bin_files[0].name if bin_files else None
                if not hd5_file_name:
                    hd5_files = [f for f in folder.glob('*.hd5')]
                    hd5_file_name = hd5_files[0].name if hd5_files else None

                df['bin_file_path'] = str(folder / bin_file_name) if bin_file_name else None
                df['hd5_file_path'] = str(folder / hd5_file_name) if hd5_file_name else None
                df['trials'] = config.get('trials')
                df['network_size'] = config.get('network', {}).get('size')
                df['op'] = config.get('op')
                df['epsilon'] = config.get('epsilon')


                result_df = pd.concat([result_df, df.dropna(axis=1, how="all")], ignore_index=True)
            except (FileNotFoundError, PermissionError, json.JSONDecodeError) as e:
                warnings.warn(f"Error processing folder {folder}: {e}", RuntimeWarning)

    return result_df



Now we need to load the simulations data. This step might take around fifteen minutes to run.

In [ ]:
sims = manual_processing(data_path)

#sims = processor(data_path)


Add some configurations we want to analyze.

The add_config function adds a new column to your existing dataframe, based on additional information it finds inside JSON configuration files.



In [ ]:
def add_config(df, key_path):
    json_cache = {}

    def get_value_from_config(config, key_path):
        keys = key_path.split('.')
        value = config
        for key in keys:
            if isinstance(value, dict):
                value = value.get(key, {})
            else:
                return None
        return value if value != {} else None

    config_values = []
    for path in df['config_json_path']:
        if path not in json_cache:
            try:
                with open(path, 'r') as f:
                    json_cache[path] = json.load(f)
            except Exception as e:
                json_cache[path] = None
        config = json_cache[path]
        if config is not None:
            config_values.append(get_value_from_config(config, key_path))
        else:
            config_values.append(None)

    column_name = key_path.replace(".", "_").replace(" ", "")
    df[column_name] = config_values

    return df



We need to add some additional data for our simulations. This might take around five minutes.

This part of the code goes through a list of specific configuration details (given in config_paths) and adds each of them as a new column to the data table.

We want to add information about:
- number of trials run in the simulation
- the size of the network
- the operation run
- the epsilon parameter (which tells us how much more effective action A is relative to action B)
- the reliability parameter
- network parameters associated with the particular network type


In [ ]:
config_paths = [
    "trials",
    "network.size",
    "op",
    "epsilon",
    "reliability",
    "network.barabasialbert.attachments",
    "network.wattsstrogatz.probability",
    "network.random.probability",
    "network.wattsstrogatz.knn"
]

for config_path in config_paths:
    sims = add_config(sims, config_path)

# Print information about the resulting DataFrame
print("\nDataFrame information:")
print(f"Shape: {sims.shape}")
print(f"Columns: {sims.columns.tolist()}")
print(f"Unique network kinds: {sims['network_kind'].unique()}")


Convert processor object to DataFrame

In [ ]:
print(sims.columns)

In [ ]:
display(sims.sample(10))


We need to extract some additional information from configuration JSON files.   
This will tell us the maximum number of steps that the simulation was run for 'max_steps' and how many simulation'repeats' were run.


In [ ]:
def get_additional_info(config_path):
  with open(config_path, 'r') as f:
        config = json.load(f)
  return {
        'max_steps': config.get('simulation', {}).get('steps'),
        'repeats': config.get('simulation', {}).get('repeats')
    }

def add_additional_info(df):
    additional_info = df['config_json_path'].apply(get_additional_info)
    df['max_steps'] = additional_info.apply(lambda x: x['max_steps'])
    df['repeats'] = additional_info.apply(lambda x: x['repeats'])
    return df


This might take up to around five minute to run.

In [ ]:
sims = add_additional_info(sims)

In [ ]:
sims = sims.fillna(0) #substituting NaN values with zeros

In [ ]:
display(sims.sample(10))


Now let's take a look in more detail at the data.

For example, what are the different network kinds?

In [ ]:
unique_network_kinds = sims['network_kind'].unique()
print("Unique network kinds:", unique_network_kinds)


What are the different Watts-Strogatz parameters that we have tested?

In [ ]:
# Filter the data for Watts-Strogatz simulations
watts_strogatz_filtered = sims[sims['network_kind'] == 'wattsstrogatz']

# Get unique values of Watts-Strogatz parameters: knn and probability
unique_knn_values = watts_strogatz_filtered['network_wattsstrogatz_knn'].unique()
unique_prob_values = watts_strogatz_filtered['network_wattsstrogatz_probability'].unique()

# Sort and display the unique values
unique_knn_values_sorted = sorted(unique_knn_values)
unique_prob_values_sorted = sorted(unique_prob_values)

print("Unique Watts-Strogatz knn values:", unique_knn_values_sorted)
print("Unique Watts-Strogatz probability values:", unique_prob_values_sorted)


How many Watts-Strogatz simulations have run for 100,000 steps, knn = 32, for each probability paramater?

In [ ]:
# Filter the data for Watts-Strogatz simulations with max steps = 100000 and knn = 32
watts_strogatz_filtered = sims[(sims['network_kind'] == 'wattsstrogatz') &
                               (sims['max_steps'] == 100000) &
                               (sims['network_wattsstrogatz_knn'] == 32)]

# Group by the 'network_wattsstrogatz_probability' column and count the occurrences
ws_simulation_counts = watts_strogatz_filtered.groupby('network_wattsstrogatz_probability').size().reset_index(name='count')

print(ws_simulation_counts)


In [ ]:
filtered_sims = sims[sims['max_steps'] == 100000]
results = []

watts_strogatz_data = filtered_sims[filtered_sims['network_kind'] == 'wattsstrogatz']
ws_grouped = watts_strogatz_data.groupby(['network_wattsstrogatz_knn', 'network_wattsstrogatz_probability']).size().reset_index(name='count')
ws_grouped['network_kind'] = 'wattsstrogatz'
results.append(ws_grouped)

barabasi_albert_data = filtered_sims[filtered_sims['network_kind'] == 'barabasialbert']
ba_grouped = barabasi_albert_data.groupby(['network_barabasialbert_attachments']).size().reset_index(name='count')
ba_grouped['network_kind'] = 'barabasialbert'
results.append(ba_grouped)

random_data = filtered_sims[filtered_sims['network_kind'] == 'random']
random_grouped = random_data.groupby(['network_random_probability']).size().reset_index(name='count')
random_grouped['network_kind'] = 'random'
results.append(random_grouped)

final_results = pd.concat(results, ignore_index=True)

display(final_results)

Now, let's create an indexer function to help us organize our simulation data based on network type and other parameters.
The indexer will take in our dataframe containing simulation data, and the network kind we want to analyze (e.g. wattsstrogatz)
It will tell us how many sims we have for each parameter, e.g. for each network kind, etc.


In [ ]:
def index_data(sims, network):
    random_conf = "network_random_probability"
    barabasi_conf = "network_barabasialbert_attachments"
    wattstrogatz_conf = "network_wattsstrogatz_probability", "network_wattsstrogatz_knn"

    df = sims.query(f"network_kind == '{network}'")

    if network == 'barabasialbert':
        df = df.drop([random_conf, *wattstrogatz_conf], axis=1)
        index_df = df.groupby(["network_kind", "network_size", "op", "epsilon", "trials", "reliability", barabasi_conf, "max_steps", "repeats"]).agg(
            input=('config_json_path', 'count'),
            output=('steps', 'count'))

    elif network == 'wattsstrogatz':
        df = df.drop([random_conf, barabasi_conf], axis=1)
        index_df = df.groupby(["network_kind", "network_size", "op", "epsilon", "trials", "reliability", *wattstrogatz_conf, "max_steps", "repeats"]).agg(
            input=('config_json_path', 'count'),
            output=('steps', 'count'))

    elif network == 'random':
        df = df.drop([*wattstrogatz_conf, barabasi_conf], axis=1)
        index_df = df.groupby(["network_kind", "network_size", "op", "epsilon", "trials", "reliability", random_conf, "max_steps", "repeats"]).agg(
            input=('config_json_path', 'count'),
            output=('steps', 'count'))

    else:
        df = df.drop([random_conf, *wattstrogatz_conf, barabasi_conf], axis=1)
        index_df = df.groupby(["network_kind", "network_size", "op", "epsilon", "trials", "reliability", "max_steps", "repeats"]).agg(
            input=('config_json_path', 'count'),
            output=('steps', 'count'))

    return index_df


Let's test our indexer on each kind of data

In [ ]:
er_index = index_data(sims, "random")
display(er_index.sample(5))

In [ ]:
ba_index = index_data(sims, "barabasialbert")
display(ba_index.head())

In [ ]:
ws_index = index_data(sims, "wattsstrogatz")
display(ws_index)

## Part 2: Exploratory Data Analysis
Now that we have our data indexed, let's explore it visually.

This will style the plots to look clear and professional.

In [ ]:
plt.style.use('seaborn')
sns.set_context("notebook", font_scale=1.2)

We want to just look at sims that ran for up to 100,000 steps.

We just want to look at parameters for which we have proper data.
That means only network_wattsstrogatz_probabilities of 0, 0.25, 0.5, 0.75

In [ ]:
filtered_sims = sims[
    (sims['max_steps'] == 100000) &
    (
        (sims['network_kind'] != 'wattsstrogatz') |
        (
            (sims['network_kind'] == 'wattsstrogatz') &  # For Watts-Strogatz, filter by specific probabilities
            (sims['network_wattsstrogatz_probability'].isin([0, 0.25, 0.5, 0.75]))
        )
    )
]

In [ ]:
filtered_sims.shape

Now, we can start with a high level overview.
We want to get a sense of how many steps it takes for the simulations  to run

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(filtered_sims['steps'])
plt.title('Distribution of Simulation Steps')
plt.xlabel('Number of Steps')
plt.ylabel('Count')
plt.yscale('log')
plt.show()


### Convergence Analysis

First, we want to explore how specific network properties affect whether the networks converge to A (the wrong result) or B (the right result).

The first of these functions calculates the percentage of different outcomes (converging to action A, B, or not converging) for specific groups of data.

The second function will be used to plot these results.

In [ ]:
def percentage_of_convergence_by_group(df, group_columns):
    results = []
    for name, group in df.groupby(group_columns):
        total = len(group)
        converged_to_B = sum((group['converged'] == True) & (group['action'] == 'B'))
        converged_to_A = sum((group['converged'] == True) & (group['action'] == 'A'))
        not_converged = total - converged_to_B - converged_to_A

        result = {
            'converged_to_B': (converged_to_B / total) * 100 if total > 0 else 0,
            'converged_to_A': (converged_to_A / total) * 100 if total > 0 else 0,
            'not_converged': (not_converged / total) * 100 if total > 0 else 0
        }
        result.update(dict(zip(group_columns, name if isinstance(name, tuple) else [name])))
        results.append(result)

    return pd.DataFrame(results)

def plot_convergence(data, x, title, ax):
    data_pivoted = data.set_index(x)[['converged_to_B', 'converged_to_A', 'not_converged']]
    data_pivoted = data_pivoted.sort_index()
    data_pivoted.plot(kind='bar', stacked=True, ax=ax, color=colors)
    ax.set_title(title, fontsize=14)
    ax.set_ylabel('Percentage', fontsize=12)
    ax.set_ylim(0, 100)
    ax.legend().remove()
    ax.set_xlabel(x, fontsize=12)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')


In [ ]:
colors = ['#1f77b4', '#2ca02c', '#d62728']

Now we perform a convergence analysis for different types of network structures based the operation.

For each network type (random, Barabási-Albert, and Watts-Strogatz), we calculate and visualizes how often nodes converge to action A, action B, or fail to converge, and displays the results in a stacked bar chart.



In [ ]:
unique_ops = filtered_sims['op'].unique()

for op_value in unique_ops:
    op_data = filtered_sims[filtered_sims['op'] == op_value]
    network_info = []

    er_sims = op_data[op_data['network_kind'] == 'random'].copy()
    if not er_sims.empty:
        er_convergence = percentage_of_convergence_by_group(er_sims, ['network_random_probability'])
        network_info.append({
            'data': er_convergence,
            'title': 'Erdős-Rényi Graphs',
            'x_var': 'network_random_probability'
        })

    ba_sims = op_data[op_data['network_kind'] == 'barabasialbert'].copy()
    if not ba_sims.empty:
        ba_convergence = percentage_of_convergence_by_group(ba_sims, ['network_barabasialbert_attachments'])
        network_info.append({
            'data': ba_convergence,
            'title': 'Barabási-Albert Graphs',
            'x_var': 'network_barabasialbert_attachments'
        })

    ws_sims = op_data[op_data['network_kind'] == 'wattsstrogatz'].copy()
    if not ws_sims.empty:
        ws_convergence = percentage_of_convergence_by_group(ws_sims, ['network_wattsstrogatz_knn'])
        network_info.append({
            'data': ws_convergence,
            'title': 'Watts-Strogatz Graphs',
            'x_var': 'network_wattsstrogatz_knn'
        })

    num_plots = len(network_info)

    if num_plots == 0:
        continue

    fig, axes = plt.subplots(num_plots, 1, figsize=(6, 6 * num_plots))
    if num_plots == 1:
        axes = [axes]

    fig.suptitle(f'Convergence Analysis (op = {op_value})', fontsize=16, y=0.98)

    for i, info in enumerate(network_info):
        plot_convergence(info['data'], info['x_var'], info['title'], axes[i])

    handles = [plt.Rectangle((0,0),1,1,color=c) for c in colors]
    labels= ['Converged to B', 'Converged to A', 'Not Converged']
    fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 0.94), ncol=3)

    plt.tight_layout(rect=[0, 0.03, 1, 0.93])
    plt.show()


Under the Bala Goyal operation, all of our networks always eventually converge to the right action, **B**.

Under the unreliable network basic gullible operation, whether the graphs converge seems to depend a lot on the network structure.

Under the unreliable negative epsilon operation, none of the (Barabasi-Albert) graphs that we have tested converge.

Watts-Strogatz has two different parameters: knn and rewiring probability. So it's worthwhile taking a more fine-grained look, comparing the convergence results for each knn for a given rewiring probability.

In [ ]:
for op_value in unique_ops:
    op_data = filtered_sims[(filtered_sims['op'] == op_value) & (filtered_sims['network_kind'] == 'wattsstrogatz')]

    if op_data.empty:
        continue

    ws_probabilities = sorted(op_data['network_wattsstrogatz_probability'].unique())
    num_plots = len(ws_probabilities)

    n_cols = 2
    n_rows = (num_plots + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 6 * n_rows))
    axes = axes.flatten()

    fig.suptitle(f'Watts-Strogatz Convergence (op = {op_value})', fontsize=16, y=0.98)

    for i, prob in enumerate(ws_probabilities):
        data_subset = op_data[op_data['network_wattsstrogatz_probability'] == prob]
        if not data_subset.empty:
            convergence_data = percentage_of_convergence_by_group(data_subset, ['network_wattsstrogatz_knn'])
            ax = axes[i]
            plot_convergence(convergence_data, 'network_wattsstrogatz_knn', f'Probability = {prob}', ax)
        else:
            axes[i].set_visible(False)

    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    handles = [plt.Rectangle((0,0),1,1,color=c) for c in colors]
    labels= ['Converged to B', 'Converged to A', 'Not Converged']
    fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 0.94), ncol=3)

    plt.tight_layout(rect=[0, 0.03, 1, 0.93])
    plt.show()


 Whilst knn seems to have a lot of effect on the convergence properties, the rewiring probability seems to have very little effect. After all,these graphs look quite similar regardless of the rewiring probability.

#### Rates of Convergence

So far, we have only looked at the final convergence properties. Do the simulations eventually converge after 100,000 steps?

But it might be interesting to see how quickly they converge -- or not.

So let's explore how specific network properties affect the number of steps in our simulations that converge.



We will draw boxplots show the distribution of the number of steps required for convergence across different simulation settings.

Alongside these, we will also show an alternative, statistical mean plots that display the average number of steps along with the error bars (standard deviation) for each setting.


In [ ]:
def create_boxplot(data, x, y, hue, ax):
    sns.boxplot(x=x, y=y, hue=hue, data=data, ax=ax, showfliers=False, palette=colors)
    ax.set_xlabel(x, fontsize=12)
    ax.set_ylabel(y, fontsize=12)
    ax.tick_params(axis='x', rotation=45)
    ax.set_ylim(10, 105000)
    ax.set_yscale('log')
    sns.despine(ax=ax)

def create_stats_plot(data, group_columns, x_axis, ax):
    step_stats = data.groupby(group_columns)['steps'].describe()
    for i, op in enumerate(data['op'].unique()):
        op_data = step_stats[step_stats.index.get_level_values('op') == op]
        ax.errorbar(op_data.index.get_level_values(x_axis),
                    op_data['mean'],
                    yerr=op_data['std'],
                    label=op,
                    marker='o',
                    capsize=5,
                    color=colors[i],
                    ecolor=colors[i],
                    linestyle='--')
    ax.set_ylim(10, 105000)
    ax.set_yscale('log')
    sns.despine(ax=ax)

def plot_network_type(data, x, title, ax1, ax2):
    create_boxplot(data, x, 'steps', 'op', ax1)
    ax1.set_title(f'Boxplot: {title}', fontsize=14)
    create_stats_plot(data, [x, "op"], x, ax2)
    ax2.set_title(f'Stats Plot: {title}', fontsize=14)
    ax2.set_xlabel(x, fontsize=12)
    ax2.set_ylabel('Number of Steps', fontsize=12)


In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(20, 30))
fig.suptitle('Network Properties vs Steps', fontsize=20, y=0.98)

er_sims = filtered_sims[filtered_sims['network_kind'] == 'random'].copy()
plot_network_type(er_sims, 'network_random_probability', 'Erdős-Rényi Graphs', axes[0, 0], axes[0, 1])

ba_sims = filtered_sims[filtered_sims['network_kind'] == 'barabasialbert'].copy()
plot_network_type(ba_sims, 'network_barabasialbert_attachments', 'Barabási-Albert Graphs', axes[1, 0], axes[1, 1])

ws_sims = filtered_sims[filtered_sims['network_kind'] == 'wattsstrogatz'].copy()
plot_network_type(ws_sims, 'network_wattsstrogatz_knn', 'Watts-Strogatz Graphs', axes[2, 0], axes[2, 1])

handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, title='Operation', loc='upper center', bbox_to_anchor=(0.5, 0.96), ncol=3)

for ax in axes.flat:
    if ax.get_legend() is not None:
        ax.get_legend().remove()

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

ws_probabilities = sorted(ws_sims['network_wattsstrogatz_probability'].unique())
n_plots = len(ws_probabilities)
n_cols = 2
n_rows = n_plots

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 10*n_rows))
fig.suptitle('Watts-Strogatz Graphs: Steps vs k_nn for Different Probabilities', fontsize=16)

for i, prob in enumerate(ws_probabilities):
    data_subset = ws_sims[ws_sims['network_wattsstrogatz_probability'] == prob]
    plot_network_type(data_subset, 'network_wattsstrogatz_knn', f'Probability = {prob}', axes[i, 0], axes[i, 1])

for ax in axes.flat:
    if ax.get_legend() is not None:
        ax.get_legend().remove()

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()


#### Rates of Convergence to the Correct Result

We might want to restrict ourselves to only simulations that converge to the correct action.

So let's explore how specific network properties affect the number of steps in our simulations that converge to the correct result (action 'B').

We want to see, for those results that converge to the correct conclusion, how quickly they do so, for different network configurations.

We'll be filtering our data to include only simulations that converged to action 'B', which represents the correct outcome in our model.


In [ ]:
sims_converged_B = filtered_sims[(filtered_sims['converged'] == True) & (sims['action'] == 'B')]

First, let's define our functions for plots.

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(20, 30))
fig.suptitle('Network Properties vs Steps for Correct Convergence', fontsize=16)

er_sims = sims_converged_B[sims_converged_B['network_kind'] == 'random'].copy()
plot_network_type(er_sims, 'network_random_probability', 'Erdős-Rényi Graphs', axes[0, 0], axes[0, 1])

ba_sims = sims_converged_B[sims_converged_B['network_kind'] == 'barabasialbert'].copy()
plot_network_type(ba_sims, 'network_barabasialbert_attachments', 'Barabási-Albert Graphs', axes[1, 0], axes[1, 1])

ws_sims = sims_converged_B[sims_converged_B['network_kind'] == 'wattsstrogatz'].copy()
plot_network_type(ws_sims, 'network_wattsstrogatz_knn', 'Watts-Strogatz Graphs', axes[2, 0], axes[2, 1])

handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, title='Operation', loc='upper center', bbox_to_anchor=(0.5, 0.96), ncol=3)

for ax in axes.flat:
    if ax.get_legend() is not None:
        ax.get_legend().remove()

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

ws_probabilities = sorted(ws_sims['network_wattsstrogatz_probability'].unique())
n_plots = len(ws_probabilities)
n_cols = 2
n_rows = n_plots

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 10*n_rows))
fig.suptitle('Watts-Strogatz Graphs: Steps vs k_nn for Different Probabilities', fontsize=16)

for i, prob in enumerate(ws_probabilities):
    data_subset = ws_sims[ws_sims['network_wattsstrogatz_probability'] == prob]
    plot_network_type(data_subset, 'network_wattsstrogatz_knn', f'Probability = {prob}', axes[i, 0], axes[i, 1])

for ax in axes.flat:
    if ax.get_legend() is not None:
        ax.get_legend().remove()

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()


In [ ]:
sims_converged_A = filtered_sims[(filtered_sims['converged'] == True) & (sims['action'] == 'A')]

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(20, 30))
fig.suptitle('Network Properties vs Steps for Incorrect Convergence', fontsize=16)

er_sims = sims_converged_A[sims_converged_A['network_kind'] == 'random'].copy()
plot_network_type(er_sims, 'network_random_probability', 'Erdős-Rényi Graphs', axes[0, 0], axes[0, 1])

ba_sims = sims_converged_A[sims_converged_A['network_kind'] == 'barabasialbert'].copy()
plot_network_type(ba_sims, 'network_barabasialbert_attachments', 'Barabási-Albert Graphs', axes[1, 0], axes[1, 1])

ws_sims = sims_converged_A[sims_converged_A['network_kind'] == 'wattsstrogatz'].copy()
plot_network_type(ws_sims, 'network_wattsstrogatz_knn', 'Watts-Strogatz Graphs', axes[2, 0], axes[2, 1])

handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, title='Operation', loc='upper center', bbox_to_anchor=(0.5, 0.96), ncol=3)

for ax in axes.flat:
    if ax.get_legend() is not None:
        ax.get_legend().remove()

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

ws_probabilities = sorted(ws_sims['network_wattsstrogatz_probability'].unique())
n_plots = len(ws_probabilities)
n_cols = 2
n_rows = n_plots

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 10*n_rows))
fig.suptitle('Watts-Strogatz Graphs: Steps vs k_nn for Different Probabilities', fontsize=16)

for i, prob in enumerate(ws_probabilities):
    data_subset = ws_sims[ws_sims['network_wattsstrogatz_probability'] == prob]
    plot_network_type(data_subset, 'network_wattsstrogatz_knn', f'Probability = {prob}', axes[i, 0], axes[i, 1])

for ax in axes.flat:
    if ax.get_legend() is not None:
        ax.get_legend().remove()

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()


### Analysis of Network Metrics

Next, let's examine how specific network properties affect the number of steps in our simulations.

1. Density
2. Longest shortest path (diameter)
3. Average local clustering coefficient
4. Global clustering coefficient
5. Average shortest path length


#### 1. Density
**Density** is a measure of how many edges are present in the network compared to the maximum number of possible edges. For a network with \( N \) nodes, the maximum number of edges is \( \frac{N(N-1)}{2} \) (for an undirected network). The density \( D \) is given by:

$$
D = \frac{\text{number of edges in the network}}{\frac{N(N-1)}{2}}
$$

Density ranges from 0 to 1, where 0 indicates a completely disconnected network (no edges), and 1 indicates a fully connected network (every node is connected to every other node).

#### 2. Longest Shortest Path (Diameter)
The **network diameter** is the maximum distance between any pair of nodes in the network. It is defined as the longest of all the shortest paths between any two nodes. The diameter represents the "longest minimum distance" that separates the most distant nodes in the network.

#### 3. Average Local Clustering Coefficient
The **average local clustering coefficient** is the average of the local clustering coefficients for all nodes in the network. The **local clustering coefficient**  measures how close a node and its neighbours are to being a complete sub-graph. It is the ratio of the number of edges between its neighbors to the total number of possible connections between those neighbors. For a node \( v \) with \( k_v \) neighbors, the local clustering coefficient \( C(v) \) is given by:

$$
C(v) = \frac{2 \times \text{number of edges between neighbors of v}}{k_v(k_v - 1)}
$$

This value ranges from 0 to 1, where 0 corresponds to no clustering and 1 corresponds to maximal clustering.

Thus the average local clusteirng coefficient indicates how likely it is that a node's neighbors are connected to each other on average. The local clustering coefficient of a node is the ratio of the number of connections between its neighbors to the number of possible connections. The average local clustering coefficient is calculated by averaging this value across all nodes in the network.

#### 4. Global Clustering Coefficient
The **global clustering coefficient**, also known as network transitivity, measures the tendency of nodes in the network to form triangles. It is calculated as the ratio of the number of closed triplets (triangles) to the total number of connected triplets (two edges that share a common node) in the network. This value reflects the overall clustering behavior of the entire network.
Mathematically:

$$
C_{\text{global}} = \frac{\text{number of closed triplets (triangles)}}{\text{number of connected triplets}}
$$

Importantly, the global coefficient **is not** simply the average of the local coefficients. The average local clustering coefficient reflects the typical clustering around individual nodes in the network. The global clustering coefficient reflects the overall tendency of the entire network to form clusters.




#### 5. Average Shortest Path Length
The **average shortest path length** is a measure of how efficiently information or influence spreads in a network. It is the average number of steps required to travel along the shortest paths between all pairs of nodes. This measure gives insight into the overall connectivity and compactness of the network. Mathematically, it's expressed as:

$$
T = \frac{3 \times \text{number of triangles}}{\text{number of triplets}}
$$

This value is also between 0 and 1, with higher values indicating a higher tendency for the network to form tight-knit groups.


### Calculating Graph metrics

First we need to calculate the relevant metrics for each graph. This might take a quite a long time.

To save some time, here we just look at the Watts-Strogatz graphs.ws_df

These calculations can be very slow, so we will also save time by doing some basic parallel processing. We use a ProcessPoolExecutor, which splits the work into multiple processes (like having several workers doing different tasks at once). This is useful for speeding up tasks that can be done independently, like calculating metrics for separate graph files.


In [ ]:
import dgl

def calculate_graph_metrics(bin_file_path):
    try:
        graphs, _ = dgl.load_graphs(bin_file_path)
        G = graphs[0]
        nx_G = G.to_networkx()
        if isinstance(nx_G, (nx.MultiGraph, nx.MultiDiGraph)):
            nx_G = nx.Graph(nx_G)
        else:
            nx_G = nx_G.to_undirected()

        density = nx.density(nx_G)
        try:
            longest_shortest_path = nx.diameter(nx_G)
        except nx.NetworkXError:
            longest_shortest_path = np.nan

        global_clustering = nx.transitivity(nx_G)
        avg_local_clustering = nx.average_clustering(nx_G)

        try:
            avg_shortest_path_length = nx.average_shortest_path_length(nx_G)
        except nx.NetworkXError:
            avg_shortest_path_length = np.nan

        return {
            'density': density,
            'longest_shortest_path': longest_shortest_path,
            'global_clustering': global_clustering,
            'avg_local_clustering': avg_local_clustering,
            'avg_shortest_path_length': avg_shortest_path_length,
            'num_nodes': nx_G.number_of_nodes(),
            'num_edges': nx_G.number_of_edges()
        }
    except Exception as e:
        print(f"Error processing {bin_file_path}: {str(e)}")
        return None



def calculate_metrics_parallel(data, num_workers=4):
    metrics = []
    with ProcessPoolExecutor(max_workers=num_workers) as executor:
        future_to_path = {executor.submit(calculate_graph_metrics, path): path for path in data['bin_file_path']}
        for future in as_completed(future_to_path):
            path = future_to_path[future]
            try:
                result = future.result()
                metrics.append((path, result))
            except Exception as e:
                print(f"Error processing {path}: {str(e)}")

    metrics_dict = {path: metric for path, metric in metrics if metric is not None}

    # Update only the rows where we successfully calculated metrics
    data['graph_metrics'] = data['bin_file_path'].map(metrics_dict)

    return data






This might take around five minutes

In [ ]:
# Apply the parallel function to the dataset
ws_sims = calculate_metrics_parallel(ws_sims)

metrics_list = ['density', 'longest_shortest_path', 'global_clustering', 'avg_local_clustering', 'avg_shortest_path_length']
for metric in metrics_list:
    ws_sims[metric] = ws_sims['graph_metrics'].apply(lambda x: x.get(metric) if x is not None else np.nan)



To plot the results, we will split the network metrics into separate bins.

In [ ]:
def bin_data(data, metric, num_bins=7, decimal_places=1):
    if metric == 'density':
        # For density, use each unique value as its own bin
        unique_values = sorted(data[metric].unique())
        value_to_bin = {val: i for i, val in enumerate(unique_values)}
        data['metric_bin'] = data[metric].map(value_to_bin)
        bin_values = unique_values  # Use unique_values as bin_values for density
    else:
        # For other metrics, use existing binning logic
        _, bin_edges = pd.qcut(data[metric], q=num_bins, retbins=True, duplicates='drop')
        rounded_edges = np.round(bin_edges, decimals=decimal_places)
        rounded_edges[0] = np.floor(bin_edges[0] * 10**decimal_places) / 10**decimal_places
        rounded_edges[-1] = np.ceil(bin_edges[-1] * 10**decimal_places) / 10**decimal_places
        rounded_edges = np.unique(rounded_edges)

        while len(rounded_edges) < 2:
            decimal_places += 1
            rounded_edges = np.round(bin_edges, decimals=decimal_places)
            rounded_edges = np.unique(rounded_edges)
            if decimal_places > 10:
                raise ValueError("Cannot create unique bin edges with current data and settings.")

        data['metric_bin'] = pd.cut(data[metric], bins=rounded_edges, include_lowest=True, right=False)
        bin_values = rounded_edges

    return data, bin_values


def plot_metric_boxplots(data, metric, bin_values):
    plt.figure(figsize=(8, 8))
    sns.boxplot(x='metric_bin', y='steps', hue='op', data=data, palette='Set2')

    plt.xlabel(metric.replace('_', ' ').title())
    plt.ylabel('Number of Steps')
    plt.yscale('log')
    plt.title(f'{metric.replace("_", " ").title()} vs Number of Steps', pad=35)

    if metric == 'density':
        plt.xticks(range(len(bin_values)), [f'{v:.4f}' for v in bin_values], rotation=90)
    else:
        plt.xticks(rotation=45, ha='right')

    plt.legend(
        loc='upper center',
        bbox_to_anchor=(0.5, 1.08),
        ncol=len(data['op'].unique()),
        fontsize='small',
        title_fontsize='small'
    )
    plt.tight_layout()
    plt.show()


In [ ]:
metrics = ['density', 'global_clustering', 'avg_local_clustering', 'avg_shortest_path_length']
for metric in metrics:
    if metric in ws_sims.columns:
        binned_data, bin_values = bin_data(ws_sims.copy(), metric, num_bins=7, decimal_places=1)
        plot_metric_boxplots(binned_data, metric, bin_values)






## Part 3: Statistical Analysis

Statistical analysis helps us to understand the significance of observed patterns and relationships in our data. It allows us to move beyond mere description to make inferences about the underlying processes that generate network structures and dynamics.

In this section, we'll focus on the **Chi-Square Test for Independence**, a technique for analyzing categorical data. This test will allow us to determine whether there are significant relationships between different features of our network simulations, such as network types, parameters, and convergence outcomes.

The Chi-Square Test for Independence is used to determine whether there is a significant relationship between two categorical variables. In our context, we'll use it to examine whether factors like network type, network parameters, or operation types are associated with convergence outcomes.

- We start with observed frequencies of outcomes (converging on action A or B) for different categories.
- We calculate expected frequencies assuming no relationship between variables.
- We compare observed and expected frequencies using the chi-square statistic.





The **p-value** derived from the chi-square test tells you the probability of observing the data (or something more extreme) under the null hypothesis. A small p-value (less than some threshold, typically 0.05) suggests rejecting the null hypothesis, meaning there's a statistically significant relationship.

I.e. a p-value less than 0.05 means that the probability of observing a difference at least this large between two variables under the null hypothesis (i.e. if there is no effect) would be less than 1/20.

-



After finding a significant result, **Cramer's V** helps you understand if the association is weak, moderate, or strong.

While chi-square only indicates if a relationship exists, Cramer's V quantifies how strong that relationship is. Cramer's V values range from 0 to 1:

- 0 means no association,
- 1 indicates a perfect association.

This is useful for understanding the practical significance of the relationship, not just whether it exists.  This is particularly helpful when working with large sample sizes, where even very weak associations might appear statistically significant.





In [ ]:
def perform_chi_square(data, group_cols, outcome_col):
    # Create a contingency table
    contingency_table = pd.crosstab(index=[data[col] for col in group_cols],
                                    columns=data[outcome_col])

    # Perform chi-square test
    chi2, p_value, dof, expected = chi2_contingency(contingency_table)

    # Calculate Cramer's V
    n = contingency_table.sum().sum()
    min_dim = min(contingency_table.shape) - 1
    cramer_v = np.sqrt(chi2 / (n * min_dim))

    return {
        'chi2': chi2,
        'p_value': p_value,
        'dof': dof,
        'cramer_v': cramer_v,
        'contingency_table': contingency_table,
        'expected': expected
    }


def categorize_convergence(row):
    if row['converged']:
        return row['action']
    else:
        return 'Not Converged'


We will filter for the network types of interest, as well as the reliability.

In [ ]:
# Filter for network types of interest
network_types_of_interest = ['random', 'wattsstrogatz', 'barabasialbert']
filtered_sims = filtered_sims[filtered_sims['network_kind'].isin(network_types_of_interest)]
# Filter the data for reliability level, as not all reliability data is available
filtered_sims = filtered_sims[
    (filtered_sims['reliability'].isin([1.0, 0.75])) |
    (filtered_sims['reliability'].isna())
]
# Add a new column for convergence category
filtered_sims['convergence_category'] = filtered_sims.apply(categorize_convergence, axis=1)


# List of comparisons to make
comparisons = [
    (['network_kind'], 'All Networks'),
    (['op'], 'All Operations'),
    (['network_kind', 'op'], 'Network Types and Operations'),
    (['network_random_probability'], 'Erdős-Rényi Probability'),
    (['network_barabasialbert_attachments'], 'Barabási-Albert Attachments'),
    (['network_wattsstrogatz_knn'], 'Watts-Strogatz k_nn'),
    (['network_wattsstrogatz_probability'], 'Watts-Strogatz Rewiring Probability'),
    (['reliability'], 'Reliability'),
    (['epsilon'], 'Epsilon')
]


Now we shall perform the chi-square tests

In [ ]:
results = []
for group_cols, description in comparisons:
    # Filter data if necessary
    if 'network_random_probability' in group_cols:
        data = filtered_sims[filtered_sims['network_kind'] == 'random']
    elif 'network_barabasialbert_attachments' in group_cols:
        data = filtered_sims[filtered_sims['network_kind'] == 'barabasialbert']
    elif 'network_wattsstrogatz' in group_cols[0]:
        data = filtered_sims[filtered_sims['network_kind'] == 'wattsstrogatz']
    else:
        data = filtered_sims

    # Create contingency table
    contingency_table = pd.crosstab(index=[data[col] for col in group_cols],
                                    columns=data['convergence_category'])

    if contingency_table.size == 0 or contingency_table.values.sum() == 0:
        skipped_main_comparisons.append({
            'Comparison': description,
            'Reason': 'Empty contingency table'
        })
        continue

    try:
        # Perform test
        result = perform_chi_square(data, group_cols, 'convergence_category')
        result['description'] = description
        result['group_cols'] = group_cols
        results.append(result)
    except ValueError as e:
        skipped_main_comparisons.append({
            'Comparison': description,
            'Reason': str(e)
        })




# Create a dataframe with the results
results_df = pd.DataFrame([{
    'Comparison': r['description'],
    'Chi-square': r['chi2'],
    'p-value': r['p_value'],
    'Degrees of Freedom': r['dof'],
    'Cramer\'s V': r['cramer_v']
} for r in results])


Now let's display  any statistically significant results -- those with a p-value below 0.05.


In [ ]:
print("Significant Chi-Square Test Results (p < 0.05):")
display(results_df[results_df['p-value'] < 0.05].sort_values('p-value'))

# Pairwise comparisons between operations
operations = filtered_sims['op'].unique()
network_types = filtered_sims['network_kind'].unique()

pairwise_results = []




### Pairwise Comparisons

So far, we have  performed chi-square tests to compare:
- Convergence outcomes across different network types
- Convergence outcomes across different operations
- Convergence outcomes for each network type across its specific parameters

This method gave us a broad overview of how different factors (network type, operation, network parameters) influenced convergence outcomes. It treated each factor independently and looked at its overall effect on convergence.

In a pairwise comparison, we look at each unique combination of network type, network parameters, and reliability.

This analysis will help us answer questions like:

- Do two different operations produce significantly different convergence outcomes in a specific type of network?
- How does this difference change as we vary network parameters or reliability?
- Are some operations more sensitive to changes in network structure than others?



For each comparison, we create a 2x2 contingency table of A and B outcomes for the two operations being compared.

We then perform a chi-square test on this contingency table to determine if there's a significant difference in the distribution of A and B outcomes between the two operations.



In [ ]:
def perform_chi_square(contingency_table):
    chi2, p_value, dof, expected = chi2_contingency(contingency_table)
    n = contingency_table.sum().sum()
    min_dim = min(contingency_table.shape) - 1
    cramer_v = np.sqrt(chi2 / (n * min_dim))
    return chi2, p_value, dof, cramer_v



In [ ]:
def analyze_contingency(data, group_cols, outcome_col):
    contingency_table = pd.crosstab(index=[data[col] for col in group_cols],
                                    columns=data[outcome_col])

    if contingency_table.empty or contingency_table.size == 0:
        return None

    if (contingency_table.sum(axis=1) == 0).any() or (contingency_table.sum(axis=0) == 0).any():
        return None

    try:
        return perform_chi_square(contingency_table)
    except ValueError:
        return None


In [ ]:
def pairwise_operation_analysis(data, network_types, operations, network_params):
    results = []
    for network_type in network_types:
        subset = data[data['network_kind'] == network_type]
        params = network_params.get(network_type, [])

        if network_type == 'wattsstrogatz':
            # For Watts-Strogatz, we need to consider both parameters together
            knn_values = sorted(subset['network_wattsstrogatz_knn'].unique())
            prob_values = sorted(subset['network_wattsstrogatz_probability'].unique())

            for knn in knn_values:
                for prob in prob_values:
                    param_subset = subset[
                        (subset['network_wattsstrogatz_knn'] == knn) &
                        (subset['network_wattsstrogatz_probability'] == prob)
                    ]

                    for op1, op2 in itertools.combinations(operations, 2):
                        result = analyze_contingency(param_subset, ['op'], 'convergence_category')
                        if result:
                            chi2, p_value, dof, cramer_v = result
                            results.append({
                                'Comparison': f'{op1} vs {op2}',
                                'Network Type': network_type,
                                'Parameters': f'k_nn={knn}, prob={prob}',
                                'Chi-square': chi2,
                                'p-value': p_value,
                                'Degrees of Freedom': dof,
                                "Cramer's V": cramer_v
                            })
        else:
            for param in params:
                param_values = sorted(subset[param].unique())

                for param_value in param_values:
                    param_subset = subset[subset[param] == param_value]

                    for op1, op2 in itertools.combinations(operations, 2):
                        result = analyze_contingency(param_subset, ['op'], 'convergence_category')
                        if result:
                            chi2, p_value, dof, cramer_v = result
                            results.append({
                                'Comparison': f'{op1} vs {op2}',
                                'Network Type': network_type,
                                'Parameters': f'{param}={param_value}',
                                'Chi-square': chi2,
                                'p-value': p_value,
                                'Degrees of Freedom': dof,
                                "Cramer's V": cramer_v
                            })
    return pd.DataFrame(results)


In [ ]:
def network_parameter_analysis(data, network_types, operations, network_params):
    results = []
    for network_type in network_types:
        subset = data[data['network_kind'] == network_type]
        params = network_params.get(network_type, [])

        for op in operations:
            op_subset = subset[subset['op'] == op]
            for param in params:
                result = analyze_contingency(op_subset, [param], 'convergence_category')
                if result:
                    chi2, p_value, dof, cramer_v = result
                    results.append({
                        'Network Type': network_type,
                        'Operation': op,
                        'Parameter': param,
                        'Chi-square': chi2,
                        'p-value': p_value,
                        'Degrees of Freedom': dof,
                        "Cramer's V": cramer_v
                    })
    return pd.DataFrame(results)





In [ ]:
def display_results(df, title, sort_by="Cramer's V"):
    print(f"\n{title}")
    display(df.sort_values(sort_by, ascending=False).reset_index(drop=True))




In [ ]:
# Setup
network_types = ['random', 'wattsstrogatz', 'barabasialbert']
operations = ['BalaGoyalOp', 'UnreliableNetworkBasicGullibleBinomialOp', 'UnreliableNetworkBasicGullibleNegativeEpsOp']
network_params = {
    'random': ['network_random_probability'],
    'wattsstrogatz': ['network_wattsstrogatz_knn', 'network_wattsstrogatz_probability'],
    'barabasialbert': ['network_barabasialbert_attachments']
}



In [ ]:
# Run analyses
parameter_results = network_parameter_analysis(filtered_sims, network_types, operations, network_params)
pairwise_results = pairwise_operation_analysis(filtered_sims, network_types, operations, network_params)



In [ ]:
for network_type in network_types:
    network_results = parameter_results[parameter_results['Network Type'] == network_type]

# Display significant results
print("\nSignificant Results (p < 0.05):")
display_results(pairwise_results[pairwise_results['p-value'] < 0.05], "Operations")
display_results(parameter_results[parameter_results['p-value'] < 0.05], "Network Parameters")



### Note on Multiple Comparisons:
- We have conducted multiple chi-square tests, which increases the risk of Type I errors (false positives).
- To address this, one could apply correction methods such as the Bonferroni correction or False Discovery Rate (FDR).
- These methods adjust p-values to maintain a family-wise error rate or control the proportion of false discoveries.
- For the purposes of this workshop, we haven't applied these corrections, but it's important to be aware of this issue in research contexts.
